In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, kruskal

print("库导入完成")


库导入完成


In [2]:
# 读取Excel文件
excel_path = r"F:\CAEEG\CAEEG Label.xlsx"

# 读取工作表（列名在第一行）
df = pd.read_excel(excel_path)

# 显示前几行和列名，了解数据结构
print("数据形状:", df.shape)
print("\n列名:")
print(df.columns.tolist())
print("\n前5行数据:")
print(df.head())
print("\n数据类型:")
print(df.dtypes)


数据形状: (111, 13)

列名:
['编号', '英文名', '性别', '年龄', '文化程度', 'Unnamed: 5', '标签 1', 'MoCA 1', 'MMSE 1', 'Unnamed: 9', '标签 2', 'MoCA 2', 'MMSE 2']

前5行数据:
   编号           英文名 性别  年龄    文化程度  Unnamed: 5 标签 1  MoCA 1  MMSE 1  \
0   1  tianxiaolong  女  78      本科         NaN  MCI      23      28   
1   2       wangyan  男  78  研究生及以上         NaN  MCI      25      29   
2   3   yuandonghui  女  80      高中         NaN   NC      29      30   
3   4     wanglijia  男  85      本科         NaN  MCI      24      26   
4   5       wangjun  女  82      中专         NaN   NC      27      26   

   Unnamed: 9 标签 2  MoCA 2  MMSE 2  
0         NaN   NC    28.0    28.0  
1         NaN   NC    27.0    29.0  
2         NaN   NC    30.0    30.0  
3         NaN   NC    27.0    26.0  
4         NaN  NaN     NaN     NaN  

数据类型:
编号              int64
英文名            object
性别             object
年龄              int64
文化程度           object
Unnamed: 5    float64
标签 1           object
MoCA 1          int64
MMSE 1          int64

In [3]:
# 检查列名结构
# 列名结构：[编号, 英文名, 性别, 年龄, 文化程度, (空), 标签 1, MoCA 1, MMSE 1, (空), 标签 2, MoCA 2, MMSE 2]
# "标签 1"为第一次测评，"标签 2"为第二次测评

print("检查列名结构...")
print("所有列名:", df.columns.tolist())
print(f"\n当前数据框形状: {df.shape}")
print(f"列头类型: 单级列头（列名在第一行）")


检查列名结构...
所有列名: ['编号', '英文名', '性别', '年龄', '文化程度', 'Unnamed: 5', '标签 1', 'MoCA 1', 'MMSE 1', 'Unnamed: 9', '标签 2', 'MoCA 2', 'MMSE 2']

当前数据框形状: (111, 13)
列头类型: 单级列头（列名在第一行）


In [4]:
# 处理数据：提取需要的列并重命名标签列
def reorganize_label_data(df):
    """
    提取基础列、标签列、量表分数列，并重命名标签列
    返回重组后的数据框
    """
    # 基础列（编号, 英文名, 性别, 年龄, 文化程度）
    base_cols = ['编号', '英文名', '性别', '年龄', '文化程度']
    
    # 获取所有列名
    available_cols = df.columns.tolist()
    
    print(f"\n处理数据...")
    print(f"可用列: {available_cols}")
    
    # 合并基础列、标签列和量表分数列
    all_cols = []
    # 添加基础列
    for base_col in base_cols:
        if base_col in available_cols:
            all_cols.append(base_col)
    # 添加标签列
    if '标签 1' in available_cols:
        all_cols.append('标签 1')
    if '标签 2' in available_cols:
        all_cols.append('标签 2')
    # 添加量表分数列
    if 'MoCA 1' in available_cols:
        all_cols.append('MoCA 1')
    if 'MMSE 1' in available_cols:
        all_cols.append('MMSE 1')
    if 'MoCA 2' in available_cols:
        all_cols.append('MoCA 2')
    if 'MMSE 2' in available_cols:
        all_cols.append('MMSE 2')
    
    div_df = df[all_cols].copy()
    
    # 重命名标签列
    rename_dict = {}
    if '标签 1' in div_df.columns:
        rename_dict['标签 1'] = '第一次测评'
    if '标签 2' in div_df.columns:
        rename_dict['标签 2'] = '第二次测评'
    
    div_df = div_df.rename(columns=rename_dict)
    
    # 计算教育年限（根据文化程度）
    def calculate_education_years(edu_level):
        """将文化程度转换为教育年限"""
        edu_mapping = {
            '小学': 6,
            '初中': 9,
            '高中': 12,
            '中专': 12,
            '大专': 15,
            '本科': 16,
            '研究生及以上': 19,
            '研究生': 19
        }
        if pd.isna(edu_level):
            return np.nan
        edu_str = str(edu_level).strip()
        return edu_mapping.get(edu_str, np.nan)
    
    if '文化程度' in div_df.columns:
        div_df['教育年限'] = div_df['文化程度'].apply(calculate_education_years)
    
    print(f"  ✓ 提取了 {len(all_cols)} 列")
    print(f"    列名: {div_df.columns.tolist()}")
    
    return div_df

# 重新组织数据
label_df = reorganize_label_data(df)
print(f"\n数据形状: {label_df.shape}")
print(f"列名: {label_df.columns.tolist()}")



处理数据...
可用列: ['编号', '英文名', '性别', '年龄', '文化程度', 'Unnamed: 5', '标签 1', 'MoCA 1', 'MMSE 1', 'Unnamed: 9', '标签 2', 'MoCA 2', 'MMSE 2']
  ✓ 提取了 11 列
    列名: ['编号', '英文名', '性别', '年龄', '文化程度', '第一次测评', '第二次测评', 'MoCA 1', 'MMSE 1', 'MoCA 2', 'MMSE 2', '教育年限']

数据形状: (111, 12)
列名: ['编号', '英文名', '性别', '年龄', '文化程度', '第一次测评', '第二次测评', 'MoCA 1', 'MMSE 1', 'MoCA 2', 'MMSE 2', '教育年限']


In [5]:
# 统计分析函数：只统计第一次测评有标签的受试者
def analyze_first_assessment(df):
    """
    对第一次测评有标签的受试者进行人口统计学分析和标签统计分析
    参数:
        df: 包含基础信息和第一次测评标签的数据框
    返回:
        统计结果字典，包含人口统计学信息和标签分析信息
    """
    results = {}
    
    # 基础过滤：必须有编号、性别、年龄
    df_base = df.dropna(subset=['编号', '性别', '年龄'])
    
    # 筛选第一次测评有标签的受试者
    if '第一次测评' in df_base.columns:
        first_assess_col = df_base['第一次测评']
        # 第一次测评有标签
        df_first = df_base[first_assess_col.notna()].copy()
    else:
        df_first = pd.DataFrame()  # 空数据框
    
    if len(df_first) == 0:
        print("警告：没有找到第一次测评有标签的受试者")
        return results
    
    # 统计人数
    results['人数'] = len(df_first)
    
    # 性别统计
    if '性别' in df_first.columns:
        gender_counts = df_first['性别'].value_counts()
        results['性别分布'] = gender_counts.to_dict()
        results['性别比例'] = (gender_counts / len(df_first) * 100).round(2).to_dict()
    
    # 年龄统计
    if '年龄' in df_first.columns:
        age_col = df_first['年龄']
        # 确保年龄是数值类型
        age_numeric = pd.to_numeric(age_col, errors='coerce')
        age_numeric = age_numeric.dropna()
        
        if len(age_numeric) > 0:
            results['年龄统计'] = {
                '均值': round(age_numeric.mean(), 2),
                '标准差': round(age_numeric.std(), 2),
                '最小值': int(age_numeric.min()),
                '最大值': int(age_numeric.max()),
                '中位数': round(age_numeric.median(), 2),
                '有效样本数': len(age_numeric)
            }
    
    # 标签统计分析
    if '第一次测评' in df_first.columns:
        # 第一次测评标签分布
        first_label_counts = df_first['第一次测评'].value_counts()
        first_label_proportions = (first_label_counts / len(df_first) * 100).round(2)
        
        # 二分类分布（将MCI和AD归类为Abnormal）
        first_label_binary = df_first['第一次测评'].replace(['MCI', 'AD'], 'Abnormal')
        first_binary_counts = first_label_binary.value_counts()
        first_binary_proportions = (first_binary_counts / len(first_label_binary) * 100).round(2)
        
        results['标签统计'] = {
            '第一次测评': {
                '原始分布': {
                    '分布': first_label_counts.to_dict(),
                    '比例(%)': first_label_proportions.to_dict()
                },
                '二分类分布': {
                    '分布': first_binary_counts.to_dict(),
                    '比例(%)': first_binary_proportions.to_dict()
                }
            }
        }
    
    # 保存df_first以便后续使用
    results['_df_first'] = df_first
    
    return results

# 对第一次测评有标签的受试者进行统计分析
print("="*80)
print("第一次测评统计分析")
print("="*80)

demo_results = analyze_first_assessment(label_df)

# 打印人口统计学统计信息
print("\n【人口统计学统计】")
for key, value in demo_results.items():
    if key in ['标签统计', '_df_first']:
        continue  # 跳过，后面单独处理
    if isinstance(value, dict):
        print(f"  {key}:")
        for k, v in value.items():
            if isinstance(v, dict):
                print(f"    {k}:")
                for k2, v2 in v.items():
                    print(f"      {k2}: {v2}")
            else:
                print(f"    {k}: {v}")
    else:
        print(f"  {key}: {value}")

# 打印标签统计信息
if '标签统计' in demo_results:
    print("\n【标签统计分析】")
    label_stats = demo_results['标签统计']
    
    # 第一次测评标签分布
    print("  第一次测评:")
    print(f"    原始分布 [NC, MCI, AD]:")
    for label, count in label_stats['第一次测评']['原始分布']['分布'].items():
        proportion = label_stats['第一次测评']['原始分布']['比例(%)'][label]
        print(f"      {label}: {count} ({proportion}%)")
    print(f"    二分类分布 [NC, Abnormal]:")
    for label, count in label_stats['第一次测评']['二分类分布']['分布'].items():
        proportion = label_stats['第一次测评']['二分类分布']['比例(%)'][label]
        print(f"      {label}: {count} ({proportion}%)")


第一次测评统计分析

【人口统计学统计】
  人数: 111
  性别分布:
    女: 76
    男: 35
  性别比例:
    女: 68.47
    男: 31.53
  年龄统计:
    均值: 78.33
    标准差: 8.2
    最小值: 55
    最大值: 95
    中位数: 80.0
    有效样本数: 111

【标签统计分析】
  第一次测评:
    原始分布 [NC, MCI, AD]:
      NC: 56 (50.45%)
      MCI: 43 (38.74%)
      AD: 12 (10.81%)
    二分类分布 [NC, Abnormal]:
      NC: 56 (50.45%)
      Abnormal: 55 (49.55%)


In [6]:
# 按类别进行人口统计学和量表分数统计分析
def analyze_by_category(df, label_col='第一次测评'):
    """
    按类别（NC, MCI, AD）进行统计分析，包括人口统计学指标和量表分数
    并进行显著性检验
    只分析第一次测评有标签的受试者
    参数:
        df: 包含所有需要分析的数据框
        label_col: 用于分组的标签列名（'第一次测评'）
    返回:
        统计结果字典
    """
    results = {}
    
    # 筛选第一次测评有标签的数据
    df_analysis = df[df[label_col].notna()].copy()
    
    if len(df_analysis) == 0:
        print(f"警告：没有找到第一次测评有标签的受试者")
        return results
    
    # 获取所有类别
    categories = sorted(df_analysis[label_col].unique())
    categories = [c for c in categories if pd.notna(c)]
    
    if len(categories) == 0:
        print(f"警告：{label_col}没有有效的类别")
        return results
    
    print(f"\n按{label_col}类别进行统计分析（仅第一次测评有标签的受试者）")
    print(f"类别: {categories}")
    print(f"总人数: {len(df_analysis)}")
    
    # 按类别分组统计
    category_stats = {}
    
    for category in categories:
        cat_df = df_analysis[df_analysis[label_col] == category]
        cat_stats = {}
        
        # 人数
        cat_stats['人数'] = len(cat_df)
        
        # 性别统计
        if '性别' in cat_df.columns:
            gender_counts = cat_df['性别'].value_counts()
            cat_stats['性别分布'] = gender_counts.to_dict()
            cat_stats['性别比例'] = (gender_counts / len(cat_df) * 100).round(2).to_dict()
        
        # 年龄统计
        if '年龄' in cat_df.columns:
            age_col = pd.to_numeric(cat_df['年龄'], errors='coerce').dropna()
            if len(age_col) > 0:
                cat_stats['年龄'] = {
                    '均值': round(age_col.mean(), 2),
                    '标准差': round(age_col.std(), 2),
                    '最小值': int(age_col.min()),
                    '最大值': int(age_col.max()),
                    '中位数': round(age_col.median(), 2),
                    '有效样本数': len(age_col)
                }
        
        # 教育年限统计
        if '教育年限' in cat_df.columns:
            edu_col = pd.to_numeric(cat_df['教育年限'], errors='coerce').dropna()
            if len(edu_col) > 0:
                cat_stats['教育年限'] = {
                    '均值': round(edu_col.mean(), 2),
                    '标准差': round(edu_col.std(), 2),
                    '最小值': int(edu_col.min()),
                    '最大值': int(edu_col.max()),
                    '中位数': round(edu_col.median(), 2),
                    '有效样本数': len(edu_col)
                }
        
        # MoCA 1统计
        if 'MoCA 1' in cat_df.columns:
            moca1_col = pd.to_numeric(cat_df['MoCA 1'], errors='coerce').dropna()
            if len(moca1_col) > 0:
                cat_stats['MoCA 1'] = {
                    '均值': round(moca1_col.mean(), 2),
                    '标准差': round(moca1_col.std(), 2),
                    '最小值': int(moca1_col.min()),
                    '最大值': int(moca1_col.max()),
                    '中位数': round(moca1_col.median(), 2),
                    '有效样本数': len(moca1_col)
                }
        
        # MMSE 1统计
        if 'MMSE 1' in cat_df.columns:
            mmse1_col = pd.to_numeric(cat_df['MMSE 1'], errors='coerce').dropna()
            if len(mmse1_col) > 0:
                cat_stats['MMSE 1'] = {
                    '均值': round(mmse1_col.mean(), 2),
                    '标准差': round(mmse1_col.std(), 2),
                    '最小值': int(mmse1_col.min()),
                    '最大值': int(mmse1_col.max()),
                    '中位数': round(mmse1_col.median(), 2),
                    '有效样本数': len(mmse1_col)
                }
        
        category_stats[category] = cat_stats
    
    results['按类别统计'] = category_stats
    
    # 显著性检验
    significance_tests = {}
    
    # 1. 性别分布：卡方检验
    if '性别' in df_analysis.columns:
        gender_contingency = pd.crosstab(df_analysis[label_col], df_analysis['性别'])
        if gender_contingency.shape[0] > 1 and gender_contingency.shape[1] > 1:
            chi2, p_value, dof, expected = chi2_contingency(gender_contingency)
            significance_tests['性别分布'] = {
                '检验方法': '卡方检验',
                '卡方值': round(chi2, 4),
                'p值': round(p_value, 4),
                '自由度': dof,
                '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
            }
    
    # 2. 年龄：ANOVA或Kruskal-Wallis检验
    if '年龄' in df_analysis.columns:
        age_groups = [pd.to_numeric(df_analysis[df_analysis[label_col] == cat]['年龄'], errors='coerce').dropna() 
                     for cat in categories]
        age_groups = [g for g in age_groups if len(g) > 0]
        if len(age_groups) >= 2:
            # 检查正态性和方差齐性
            try:
                f_stat, p_value = f_oneway(*age_groups)
                significance_tests['年龄'] = {
                    '检验方法': '单因素方差分析(ANOVA)',
                    'F值': round(f_stat, 4),
                    'p值': round(p_value, 4),
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
            except:
                # 如果ANOVA失败，使用Kruskal-Wallis检验
                h_stat, p_value = kruskal(*age_groups)
                significance_tests['年龄'] = {
                    '检验方法': 'Kruskal-Wallis检验',
                    'H值': round(h_stat, 4),
                    'p值': round(p_value, 4),
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
    
    # 3. 教育年限：ANOVA或Kruskal-Wallis检验
    if '教育年限' in df_analysis.columns:
        edu_groups = [pd.to_numeric(df_analysis[df_analysis[label_col] == cat]['教育年限'], errors='coerce').dropna() 
                     for cat in categories]
        edu_groups = [g for g in edu_groups if len(g) > 0]
        if len(edu_groups) >= 2:
            try:
                f_stat, p_value = f_oneway(*edu_groups)
                significance_tests['教育年限'] = {
                    '检验方法': '单因素方差分析(ANOVA)',
                    'F值': round(f_stat, 4),
                    'p值': round(p_value, 4),
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
            except:
                h_stat, p_value = kruskal(*edu_groups)
                significance_tests['教育年限'] = {
                    '检验方法': 'Kruskal-Wallis检验',
                    'H值': round(h_stat, 4),
                    'p值': round(p_value, 4),
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
    
    # 4. MoCA 1：ANOVA或Kruskal-Wallis检验
    if 'MoCA 1' in df_analysis.columns:
        moca1_groups = [pd.to_numeric(df_analysis[df_analysis[label_col] == cat]['MoCA 1'], errors='coerce').dropna() 
                       for cat in categories]
        moca1_groups = [g for g in moca1_groups if len(g) > 0]
        if len(moca1_groups) >= 2:
            try:
                f_stat, p_value = f_oneway(*moca1_groups)
                significance_tests['MoCA 1'] = {
                    '检验方法': '单因素方差分析(ANOVA)',
                    'F值': round(f_stat, 4),
                    'p值': round(p_value, 4),
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
            except:
                h_stat, p_value = kruskal(*moca1_groups)
                significance_tests['MoCA 1'] = {
                    '检验方法': 'Kruskal-Wallis检验',
                    'H值': round(h_stat, 4),
                    'p值': round(p_value, 4),
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
    
    # 5. MMSE 1：ANOVA或Kruskal-Wallis检验
    if 'MMSE 1' in df_analysis.columns:
        mmse1_groups = [pd.to_numeric(df_analysis[df_analysis[label_col] == cat]['MMSE 1'], errors='coerce').dropna() 
                       for cat in categories]
        mmse1_groups = [g for g in mmse1_groups if len(g) > 0]
        if len(mmse1_groups) >= 2:
            try:
                f_stat, p_value = f_oneway(*mmse1_groups)
                significance_tests['MMSE 1'] = {
                    '检验方法': '单因素方差分析(ANOVA)',
                    'F值': round(f_stat, 4),
                    'p值': round(p_value, 4),
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
            except:
                h_stat, p_value = kruskal(*mmse1_groups)
                significance_tests['MMSE 1'] = {
                    '检验方法': 'Kruskal-Wallis检验',
                    'H值': round(h_stat, 4),
                    'p值': round(p_value, 4),
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
    
    results['显著性检验'] = significance_tests
    
    # 二分类分析（NC vs Abnormal）
    # 创建二分类标签列
    df_analysis_binary = df_analysis.copy()
    df_analysis_binary['二分类标签'] = df_analysis_binary[label_col].replace(['MCI', 'AD'], 'Abnormal')
    
    # 获取二分类类别
    binary_categories = sorted(df_analysis_binary['二分类标签'].unique())
    binary_categories = [c for c in binary_categories if pd.notna(c)]
    
    if len(binary_categories) >= 2:
        print(f"\n按二分类（NC vs Abnormal）进行统计分析")
        print(f"类别: {binary_categories}")
        print(f"总人数: {len(df_analysis_binary)}")
        
        # 按二分类分组统计
        binary_category_stats = {}
        
        for category in binary_categories:
            cat_df = df_analysis_binary[df_analysis_binary['二分类标签'] == category]
            cat_stats = {}
            
            # 人数
            cat_stats['人数'] = len(cat_df)
            
            # 性别统计
            if '性别' in cat_df.columns:
                gender_counts = cat_df['性别'].value_counts()
                cat_stats['性别分布'] = gender_counts.to_dict()
                cat_stats['性别比例'] = (gender_counts / len(cat_df) * 100).round(2).to_dict()
            
            # 年龄统计
            if '年龄' in cat_df.columns:
                age_col = pd.to_numeric(cat_df['年龄'], errors='coerce').dropna()
                if len(age_col) > 0:
                    cat_stats['年龄'] = {
                        '均值': round(age_col.mean(), 2),
                        '标准差': round(age_col.std(), 2),
                        '最小值': int(age_col.min()),
                        '最大值': int(age_col.max()),
                        '中位数': round(age_col.median(), 2),
                        '有效样本数': len(age_col)
                    }
            
            # 教育年限统计
            if '教育年限' in cat_df.columns:
                edu_col = pd.to_numeric(cat_df['教育年限'], errors='coerce').dropna()
                if len(edu_col) > 0:
                    cat_stats['教育年限'] = {
                        '均值': round(edu_col.mean(), 2),
                        '标准差': round(edu_col.std(), 2),
                        '最小值': int(edu_col.min()),
                        '最大值': int(edu_col.max()),
                        '中位数': round(edu_col.median(), 2),
                        '有效样本数': len(edu_col)
                    }
            
            # MoCA 1统计
            if 'MoCA 1' in cat_df.columns:
                moca1_col = pd.to_numeric(cat_df['MoCA 1'], errors='coerce').dropna()
                if len(moca1_col) > 0:
                    cat_stats['MoCA 1'] = {
                        '均值': round(moca1_col.mean(), 2),
                        '标准差': round(moca1_col.std(), 2),
                        '最小值': int(moca1_col.min()),
                        '最大值': int(moca1_col.max()),
                        '中位数': round(moca1_col.median(), 2),
                        '有效样本数': len(moca1_col)
                    }
            
            # MMSE 1统计
            if 'MMSE 1' in cat_df.columns:
                mmse1_col = pd.to_numeric(cat_df['MMSE 1'], errors='coerce').dropna()
                if len(mmse1_col) > 0:
                    cat_stats['MMSE 1'] = {
                        '均值': round(mmse1_col.mean(), 2),
                        '标准差': round(mmse1_col.std(), 2),
                        '最小值': int(mmse1_col.min()),
                        '最大值': int(mmse1_col.max()),
                        '中位数': round(mmse1_col.median(), 2),
                        '有效样本数': len(mmse1_col)
                    }
            
            binary_category_stats[category] = cat_stats
        
        results['二分类统计'] = binary_category_stats
        
        # 二分类显著性检验
        binary_significance_tests = {}
        
        # 1. 性别分布：卡方检验
        if '性别' in df_analysis_binary.columns:
            gender_contingency = pd.crosstab(df_analysis_binary['二分类标签'], df_analysis_binary['性别'])
            if gender_contingency.shape[0] > 1 and gender_contingency.shape[1] > 1:
                chi2, p_value, dof, expected = chi2_contingency(gender_contingency)
                binary_significance_tests['性别分布'] = {
                    '检验方法': '卡方检验',
                    '卡方值': round(chi2, 4),
                    'p值': round(p_value, 4),
                    '自由度': dof,
                    '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                }
        
        # 2. 年龄：t检验或Mann-Whitney U检验
        if '年龄' in df_analysis_binary.columns:
            age_groups = [pd.to_numeric(df_analysis_binary[df_analysis_binary['二分类标签'] == cat]['年龄'], errors='coerce').dropna() 
                         for cat in binary_categories]
            age_groups = [g for g in age_groups if len(g) > 0]
            if len(age_groups) == 2:
                try:
                    t_stat, p_value = stats.ttest_ind(age_groups[0], age_groups[1])
                    binary_significance_tests['年龄'] = {
                        '检验方法': '独立样本t检验',
                        't值': round(t_stat, 4),
                        'p值': round(p_value, 4),
                        '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                    }
                except:
                    u_stat, p_value = stats.mannwhitneyu(age_groups[0], age_groups[1], alternative='two-sided')
                    binary_significance_tests['年龄'] = {
                        '检验方法': 'Mann-Whitney U检验',
                        'U值': round(u_stat, 4),
                        'p值': round(p_value, 4),
                        '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                    }
        
        # 3. 教育年限：t检验或Mann-Whitney U检验
        if '教育年限' in df_analysis_binary.columns:
            edu_groups = [pd.to_numeric(df_analysis_binary[df_analysis_binary['二分类标签'] == cat]['教育年限'], errors='coerce').dropna() 
                         for cat in binary_categories]
            edu_groups = [g for g in edu_groups if len(g) > 0]
            if len(edu_groups) == 2:
                try:
                    t_stat, p_value = stats.ttest_ind(edu_groups[0], edu_groups[1])
                    binary_significance_tests['教育年限'] = {
                        '检验方法': '独立样本t检验',
                        't值': round(t_stat, 4),
                        'p值': round(p_value, 4),
                        '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                    }
                except:
                    u_stat, p_value = stats.mannwhitneyu(edu_groups[0], edu_groups[1], alternative='two-sided')
                    binary_significance_tests['教育年限'] = {
                        '检验方法': 'Mann-Whitney U检验',
                        'U值': round(u_stat, 4),
                        'p值': round(p_value, 4),
                        '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                    }
        
        # 4. MoCA 1：t检验或Mann-Whitney U检验
        if 'MoCA 1' in df_analysis_binary.columns:
            moca1_groups = [pd.to_numeric(df_analysis_binary[df_analysis_binary['二分类标签'] == cat]['MoCA 1'], errors='coerce').dropna() 
                           for cat in binary_categories]
            moca1_groups = [g for g in moca1_groups if len(g) > 0]
            if len(moca1_groups) == 2:
                try:
                    t_stat, p_value = stats.ttest_ind(moca1_groups[0], moca1_groups[1])
                    binary_significance_tests['MoCA 1'] = {
                        '检验方法': '独立样本t检验',
                        't值': round(t_stat, 4),
                        'p值': round(p_value, 4),
                        '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                    }
                except:
                    u_stat, p_value = stats.mannwhitneyu(moca1_groups[0], moca1_groups[1], alternative='two-sided')
                    binary_significance_tests['MoCA 1'] = {
                        '检验方法': 'Mann-Whitney U检验',
                        'U值': round(u_stat, 4),
                        'p值': round(p_value, 4),
                        '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                    }
        
        # 5. MMSE 1：t检验或Mann-Whitney U检验
        if 'MMSE 1' in df_analysis_binary.columns:
            mmse1_groups = [pd.to_numeric(df_analysis_binary[df_analysis_binary['二分类标签'] == cat]['MMSE 1'], errors='coerce').dropna() 
                           for cat in binary_categories]
            mmse1_groups = [g for g in mmse1_groups if len(g) > 0]
            if len(mmse1_groups) == 2:
                try:
                    t_stat, p_value = stats.ttest_ind(mmse1_groups[0], mmse1_groups[1])
                    binary_significance_tests['MMSE 1'] = {
                        '检验方法': '独立样本t检验',
                        't值': round(t_stat, 4),
                        'p值': round(p_value, 4),
                        '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                    }
                except:
                    u_stat, p_value = stats.mannwhitneyu(mmse1_groups[0], mmse1_groups[1], alternative='two-sided')
                    binary_significance_tests['MMSE 1'] = {
                        '检验方法': 'Mann-Whitney U检验',
                        'U值': round(u_stat, 4),
                        'p值': round(p_value, 4),
                        '显著性': 'p < 0.05' if p_value < 0.05 else 'p >= 0.05'
                    }
        
        results['二分类显著性检验'] = binary_significance_tests
    
    return results

# 对第一次测评按类别进行统计分析
print("="*80)
print("第一次测评 - 按类别统计分析")
print("="*80)

category_results_1 = analyze_by_category(label_df, label_col='第一次测评')

# 打印按类别统计结果
if '按类别统计' in category_results_1:
    print("\n【按类别统计结果】")
    for category, stats_dict in category_results_1['按类别统计'].items():
        print(f"\n{category}组 (n={stats_dict.get('人数', 'N/A')}):")
        
        # 性别分布
        if '性别分布' in stats_dict:
            print(f"  性别分布:")
            for gender, count in stats_dict['性别分布'].items():
                proportion = stats_dict['性别比例'][gender]
                print(f"    {gender}: {count} ({proportion}%)")
        
        # 年龄
        if '年龄' in stats_dict:
            age_stats = stats_dict['年龄']
            print(f"  年龄: {age_stats['均值']} ± {age_stats['标准差']} (范围: {age_stats['最小值']}-{age_stats['最大值']}, 中位数: {age_stats['中位数']})")
        
        # 教育年限
        if '教育年限' in stats_dict:
            edu_stats = stats_dict['教育年限']
            print(f"  教育年限: {edu_stats['均值']} ± {edu_stats['标准差']} (范围: {edu_stats['最小值']}-{edu_stats['最大值']}, 中位数: {edu_stats['中位数']})")
        
        # MoCA 1
        if 'MoCA 1' in stats_dict:
            moca1_stats = stats_dict['MoCA 1']
            print(f"  MoCA 1: {moca1_stats['均值']} ± {moca1_stats['标准差']} (范围: {moca1_stats['最小值']}-{moca1_stats['最大值']}, 中位数: {moca1_stats['中位数']})")
        
        # MMSE 1
        if 'MMSE 1' in stats_dict:
            mmse1_stats = stats_dict['MMSE 1']
            print(f"  MMSE 1: {mmse1_stats['均值']} ± {mmse1_stats['标准差']} (范围: {mmse1_stats['最小值']}-{mmse1_stats['最大值']}, 中位数: {mmse1_stats['中位数']})")

# 打印显著性检验结果
if '显著性检验' in category_results_1:
    print("\n【显著性检验结果】")
    for indicator, test_result in category_results_1['显著性检验'].items():
        print(f"\n{indicator}:")
        print(f"  检验方法: {test_result['检验方法']}")
        if 'F值' in test_result:
            print(f"  F值: {test_result['F值']}")
        elif 'H值' in test_result:
            print(f"  H值: {test_result['H值']}")
        elif '卡方值' in test_result:
            print(f"  卡方值: {test_result['卡方值']}")
        print(f"  p值: {test_result['p值']}")
        print(f"  显著性: {test_result['显著性']}")

# 打印二分类统计结果
if '二分类统计' in category_results_1:
    print("\n" + "="*80)
    print("第一次测评 - 二分类统计分析（NC vs Abnormal）")
    print("="*80)
    print("\n【按二分类统计结果】")
    for category, stats_dict in category_results_1['二分类统计'].items():
        print(f"\n{category}组 (n={stats_dict.get('人数', 'N/A')}):")
        
        # 性别分布
        if '性别分布' in stats_dict:
            print(f"  性别分布:")
            for gender, count in stats_dict['性别分布'].items():
                proportion = stats_dict['性别比例'][gender]
                print(f"    {gender}: {count} ({proportion}%)")
        
        # 年龄
        if '年龄' in stats_dict:
            age_stats = stats_dict['年龄']
            print(f"  年龄: {age_stats['均值']} ± {age_stats['标准差']} (范围: {age_stats['最小值']}-{age_stats['最大值']}, 中位数: {age_stats['中位数']})")
        
        # 教育年限
        if '教育年限' in stats_dict:
            edu_stats = stats_dict['教育年限']
            print(f"  教育年限: {edu_stats['均值']} ± {edu_stats['标准差']} (范围: {edu_stats['最小值']}-{edu_stats['最大值']}, 中位数: {edu_stats['中位数']})")
        
        # MoCA 1
        if 'MoCA 1' in stats_dict:
            moca1_stats = stats_dict['MoCA 1']
            print(f"  MoCA 1: {moca1_stats['均值']} ± {moca1_stats['标准差']} (范围: {moca1_stats['最小值']}-{moca1_stats['最大值']}, 中位数: {moca1_stats['中位数']})")
        
        # MMSE 1
        if 'MMSE 1' in stats_dict:
            mmse1_stats = stats_dict['MMSE 1']
            print(f"  MMSE 1: {mmse1_stats['均值']} ± {mmse1_stats['标准差']} (范围: {mmse1_stats['最小值']}-{mmse1_stats['最大值']}, 中位数: {mmse1_stats['中位数']})")

# 打印二分类显著性检验结果
if '二分类显著性检验' in category_results_1:
    print("\n【二分类显著性检验结果】")
    for indicator, test_result in category_results_1['二分类显著性检验'].items():
        print(f"\n{indicator}:")
        print(f"  检验方法: {test_result['检验方法']}")
        if 't值' in test_result:
            print(f"  t值: {test_result['t值']}")
        elif 'U值' in test_result:
            print(f"  U值: {test_result['U值']}")
        elif '卡方值' in test_result:
            print(f"  卡方值: {test_result['卡方值']}")
        print(f"  p值: {test_result['p值']}")
        print(f"  显著性: {test_result['显著性']}")




第一次测评 - 按类别统计分析

按第一次测评类别进行统计分析（仅第一次测评有标签的受试者）
类别: ['AD', 'MCI', 'NC']
总人数: 111

按二分类（NC vs Abnormal）进行统计分析
类别: ['Abnormal', 'NC']
总人数: 111

【按类别统计结果】

AD组 (n=12):
  性别分布:
    女: 7 (58.33%)
    男: 5 (41.67%)
  年龄: 83.75 ± 7.44 (范围: 64-95, 中位数: 84.0)
  教育年限: 11.17 ± 3.69 (范围: 6-16, 中位数: 10.5)
  MoCA 1: 13.42 ± 3.09 (范围: 6-17, 中位数: 14.0)
  MMSE 1: 18.08 ± 4.68 (范围: 8-25, 中位数: 17.5)

MCI组 (n=43):
  性别分布:
    女: 25 (58.14%)
    男: 18 (41.86%)
  年龄: 79.0 ± 7.58 (范围: 61-93, 中位数: 79.0)
  教育年限: 13.93 ± 3.52 (范围: 6-19, 中位数: 16.0)
  MoCA 1: 21.91 ± 2.54 (范围: 17-25, 中位数: 22.0)
  MMSE 1: 25.26 ± 3.13 (范围: 13-29, 中位数: 26.0)

NC组 (n=56):
  性别分布:
    女: 44 (78.57%)
    男: 12 (21.43%)
  年龄: 76.66 ± 8.37 (范围: 55-94, 中位数: 78.0)
  教育年限: 13.95 ± 2.24 (范围: 9-16, 中位数: 15.0)
  MoCA 1: 27.46 ± 1.58 (范围: 25-30, 中位数: 27.0)
  MMSE 1: 28.55 ± 1.33 (范围: 26-30, 中位数: 28.5)

【显著性检验结果】

性别分布:
  检验方法: 卡方检验
  卡方值: 5.3435
  p值: 0.0691
  显著性: p >= 0.05

年龄:
  检验方法: 单因素方差分析(ANOVA)
  F值: 4.1487
  p值: 0.0184
  显著性: p < 0.05


In [7]:
# 将第一次测评统计分析结果保存到Excel
from datetime import datetime
import os

def save_binary_analysis_to_excel(category_results_1, output_dir="F:\\CAEEG"):
    """
    将第一次测评统计分析结果整理并保存到Excel文件
    参数:
        category_results_1: 第一次测评的统计结果
        output_dir: 输出目录
    """
    # 确保输出目录存在
    os.makedirs(output_dir, exist_ok=True)
    
    # 创建Excel写入器
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    excel_path = os.path.join(output_dir, f"第一次测评统计分析结果_{timestamp}.xlsx")
    writer = pd.ExcelWriter(excel_path, engine='openpyxl')
    
    # 1. 第一次测评 - 按类别统计
    if '二分类统计' in category_results_1:
        data_list = []
        for category, stats_dict in category_results_1['二分类统计'].items():
            row = {'类别': category, '测评': '第一次测评'}
            row['人数'] = stats_dict.get('人数', '')
            
            # 性别分布
            if '性别分布' in stats_dict:
                for gender, count in stats_dict['性别分布'].items():
                    proportion = stats_dict['性别比例'][gender]
                    row[f'性别_{gender}_人数'] = count
                    row[f'性别_{gender}_比例(%)'] = proportion
            
            # 年龄
            if '年龄' in stats_dict:
                age_stats = stats_dict['年龄']
                row['年龄_均值'] = age_stats['均值']
                row['年龄_标准差'] = age_stats['标准差']
                row['年龄_最小值'] = age_stats['最小值']
                row['年龄_最大值'] = age_stats['最大值']
                row['年龄_中位数'] = age_stats['中位数']
            
            # 教育年限
            if '教育年限' in stats_dict:
                edu_stats = stats_dict['教育年限']
                row['教育年限_均值'] = edu_stats['均值']
                row['教育年限_标准差'] = edu_stats['标准差']
                row['教育年限_最小值'] = edu_stats['最小值']
                row['教育年限_最大值'] = edu_stats['最大值']
                row['教育年限_中位数'] = edu_stats['中位数']
            
            # MoCA 1
            if 'MoCA 1' in stats_dict:
                moca1_stats = stats_dict['MoCA 1']
                row['MoCA1_均值'] = moca1_stats['均值']
                row['MoCA1_标准差'] = moca1_stats['标准差']
                row['MoCA1_最小值'] = moca1_stats['最小值']
                row['MoCA1_最大值'] = moca1_stats['最大值']
                row['MoCA1_中位数'] = moca1_stats['中位数']
            
            # MMSE 1
            if 'MMSE 1' in stats_dict:
                mmse1_stats = stats_dict['MMSE 1']
                row['MMSE1_均值'] = mmse1_stats['均值']
                row['MMSE1_标准差'] = mmse1_stats['标准差']
                row['MMSE1_最小值'] = mmse1_stats['最小值']
                row['MMSE1_最大值'] = mmse1_stats['最大值']
                row['MMSE1_中位数'] = mmse1_stats['中位数']
            
            data_list.append(row)
        
        df_stats_1 = pd.DataFrame(data_list)
        df_stats_1.to_excel(writer, sheet_name='第一次测评_统计', index=False)
    
    # 2. 第一次测评 - 显著性检验
    if '二分类显著性检验' in category_results_1:
        test_list = []
        for indicator, test_result in category_results_1['二分类显著性检验'].items():
            row = {'指标': indicator, '测评': '第一次测评'}
            row['检验方法'] = test_result['检验方法']
            if 't值' in test_result:
                row['统计量'] = test_result['t值']
            elif 'U值' in test_result:
                row['统计量'] = test_result['U值']
            elif '卡方值' in test_result:
                row['统计量'] = test_result['卡方值']
            row['p值'] = test_result['p值']
            row['显著性'] = test_result['显著性']
            test_list.append(row)
        
        df_test_1 = pd.DataFrame(test_list)
        df_test_1.to_excel(writer, sheet_name='第一次测评_显著性检验', index=False)
    
    # 保存Excel文件
    writer.close()
    print(f"\n第一次测评统计分析结果已保存到: {excel_path}")
    return excel_path

# 保存第一次测评统计分析结果
if '二分类统计' in category_results_1:
    excel_file = save_binary_analysis_to_excel(category_results_1)
    print(f"Excel文件路径: {excel_file}")
else:
    print("警告：没有找到第一次测评统计结果，无法保存到Excel")



第一次测评统计分析结果已保存到: F:\CAEEG\第一次测评统计分析结果_20260114_230024.xlsx
Excel文件路径: F:\CAEEG\第一次测评统计分析结果_20260114_230024.xlsx
